# OnSSET → OnStove Linking Pipeline

This notebook takes OnSSET electrification results (settlement-level CSV) and
prepares them as rasterised inputs for OnStove clean-cooking analysis.

**Pipeline steps:**
1. Load OnSSET results and settlement cluster geometries
2. Reclassify electrification codes (only grid and mini-grid support cooking)
3. Compute settlement-level LCOE and electrification rates
4. Merge tabular results with spatial cluster geometries
5. Rasterise key attributes for OnStove ingestion

## 1. Configuration

Set the paths below to match your local file structure before running.  
All other cells use these variables — no other paths are hardcoded.

In [ ]:
# ── Paths: edit these to match your local setup ──────────────────────────

# OnSSET results CSV (settlement-level output)
ONSSET_CSV = "data/input/KEN-1-2_0.csv"

# Settlement cluster geometries (shapefile or geopackage)
CLUSTERS_SHP = "data/input/clusters.shp"

# Directory where rasterised outputs will be saved
OUTPUT_DIR = "data/output"

# ── Parameters ──────────────────────────────────────────────────────────

# OnSSET modelling time horizon
START_YEAR = 2023
END_YEAR = 2033

# Raster cell size in metres (for rasterisation step)
CELL_SIZE = 100

# Target CRS (EPSG:3395 = World Mercator, used by OnStove)
TARGET_CRS = 3395

## 2. Imports

In [ ]:
import os

import pandas as pd
import geopandas as gpd

from onstove import VectorLayer

## 3. Load OnSSET results

In [ ]:
edf = pd.read_csv(ONSSET_CSV)

# Drop any stale geometry column from the CSV (spatial info comes from clusters)
edf.drop(columns="geometry", inplace=True, errors="ignore")

print(f"Loaded {len(edf):,} settlements")
print(f"Base-year population: {edf['PopStartYear'].sum():,.0f}")
print(f"Base-year electrification rate: "
      f"{edf['ElecPopCalib'].sum() / edf['PopStartYear'].sum():.1%}")

## 4. Reclassify electrification codes

OnSSET assigns technology codes that include standalone solar (code 99) and
standalone diesel (code 3). For cooking-energy analysis, only **grid** and
**mini-grid** connections can realistically power electric cooking appliances,
so we reclassify those codes to 0 (unelectrified for cooking purposes).

In [ ]:
years = range(START_YEAR, END_YEAR + 1)

for year in years:
    # Zero out energy where settlement is not electrified
    mask_unelec = edf[f"ElecStatusIn{year}"] == 0
    edf.loc[mask_unelec, f"EnergyPerSettlement{year}"] = 0

    # Adjust LCOE for standalone systems (code offset of 99)
    mask_sa = edf[f"MinimumOverallLCOE{year}"] > 99
    edf.loc[mask_sa, f"MinimumOverallLCOE{year}"] -= 99

    # Binary reclassification: grid/mini-grid → 1, standalone/unelectrified → 0
    edf[f"FinalElecCode{year}"] = edf[f"FinalElecCode{year}"].apply(
        lambda x: 0 if x in [99, 3] else 1
    )

## 5. Compute settlement-level LCOE

Weighted average LCOE across the modelling horizon, using each year's
energy demand as the weight.

In [ ]:
edf["EnergyPerSettlement"] = edf.filter(like="EnergyPerSettlement20").sum(axis=1)

edf["ecost"] = sum(
    edf[f"EnergyPerSettlement{y}"] * edf[f"MinimumOverallLCOE{y}"]
    for y in years
)

edf["setLCOE"] = edf["ecost"] / edf["EnergyPerSettlement"]
edf["setLCOE"] = edf["setLCOE"].fillna(0)

## 6. Compute electrification rate per settlement

In [ ]:
edf["ElecRate"] = edf["ElecPopCalib"] / edf["PopStartYear"]
edf["ElecRate"] = edf["ElecRate"].clip(0, 1)  # guard against edge cases

## 7. Merge with cluster geometries

Join the tabular OnSSET results to the spatial settlement clusters
on their shared `id` column, then reproject to the target CRS.

In [ ]:
clusters = gpd.read_file(CLUSTERS_SHP)
cluster_geom = clusters[["id", "geometry"]].copy()

merged = cluster_geom.merge(edf, on="id", how="inner")
merged = merged.to_crs(epsg=TARGET_CRS)

print(f"Merged settlements: {len(merged):,}")
print(f"Target-year electrified population: "
      f"{merged.loc[merged['FinalElecCode2033'] == 1, 'Pop2033'].sum():,.0f} "
      f"/ {merged['Pop2033'].sum():,.0f}")

## 8. Export electrification vector layer (optional)

In [ ]:
os.makedirs(os.path.join(OUTPUT_DIR, "onsset"), exist_ok=True)

merged[["id", "FinalElecCode2033", "ElecRate", "geometry"]].to_file(
    os.path.join(OUTPUT_DIR, "onsset", "onsset_elec.gpkg"), driver="GPKG"
)

print("Saved electrification vector to onsset_elec.gpkg")

## 9. Rasterise for OnStove

Convert the merged vector data into raster layers that OnStove can ingest.
Each attribute becomes a separate GeoTIFF at the specified cell size.

In [ ]:
# Wrap the merged GeoDataFrame in an OnStove VectorLayer
onsset = VectorLayer()
onsset.data = merged
onsset.reproject(TARGET_CRS)

# Attributes to rasterise
raster_attrs = (
    [f"FinalElecCode{y}" for y in years]
    + ["setLCOE", "ElecRate", f"Pop{START_YEAR}", f"Pop{END_YEAR}"]
)

for attr in raster_attrs:
    raster = onsset.rasterize(
        cell_height=CELL_SIZE, cell_width=CELL_SIZE, attribute=attr
    )
    raster.name = attr
    raster.save(os.path.join(OUTPUT_DIR, "onsset"))
    print(f"  ✓ {attr}.tif")

print(f"\nSaved {len(raster_attrs)} rasters to {OUTPUT_DIR}/onsset/")